<a href="https://colab.research.google.com/github/avaabarnes/A03-knn/blob/main/notebook_(3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Assignment 3: $k$ Nearest Neighbor


**Q1.**
1. What is the difference between regression and classification?
2. What is a confusion table? What does it help us understand about a model's performance?
3. What does the SSE quantify about a particular model?
4. What are overfitting and underfitting?
5. Why does splitting the data into training and testing sets, and choosing $k$ by evaluating accuracy or SSE on the test set, improve model performance?
6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.

**Q2.** This is a case study on $k$ nearest neighbor classification, using the `land_mines.csv` data.

The data consists of a label, `mine_type`, taking integer values 1 to 5, and three properties of the mine, `voltage`, `height` and `soil`. We want to predict the kind of mine from data about it. Imagine working for the DOD or a humanitarian aid agency, trying to help people remove land mines more safely.

1. Load the data. Perform some EDA, summarizing the target label and the features.
2. Split the sample 50/50 into training and test/validation sets. (The smaller the data are, the more equal the split should be, in my experience: Otherwise, all of the members of one class end up in the training or test data, and the model falls apart.)
3. Build a $k$-NN classifier. Explain how you select $k$.
4. Print a confusion table for the optimal model, comparing predicted and actual class label on the test set. How accurate is it? Where is performance more or less accurate?
5. Notice that you can have a lot of accurate predictions for a given type of mine, but still make a lot of mistakes. Please explain how you'd advise someone to actually use this predictive model in practice, given the errors that it tends to make.

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** Ava Barnes

**Date:** 9/20/2026

**Q1. Answers**
1. What is the difference between regression and classification?

Regressions is used to predict a numeric tagret, like price, and classification is to predict a categorical variable,like color.  

2. What is a confusion table? What does it help us understand about a model's performance?

A confusion table counts predictions by actual class and predicted class. It helps us understand how often a model correctly and incorrectly classifies each class.



3. What does the SSE quantify about a particular model?

The Sum of Squared Error measures the total squared difference between actual values and the model's predictions for a numeric target. A lower validation SSE means smaller errors in the validation set and higher means more errors.

4. What are overfitting and underfitting?

Overfitting occurs when a model is too complex, learning noise or specific patterns rather than general patterns from the training data. This can happen in a k-NN if that k value is very small. Underfitting occurs when a model over simplifies the data rather than capturing important patterns beause the k value is too large. Underfitting causes the model to make predictions off of too many nieghbors.


5. Why does splitting the data into training and testing sets, and choosing $k$ by evaluating accuracy or SSE on the test set, improve model performance?


We fit models on a training set and compare their predictions on a separate validation set the models have not seen. The validation error estimates how a model will perform on future, unknown data, and it lets us pick the k that generalizes best rather than the k that memorizes best. Since choosing k on the validation set makes its accuracy slightly optimistic, we keep a separate final test set for the last evaluation when one is available. This estimate depends on how we split the data.


6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.

A class label gives one clear answer that is simple to interpret, but it hides the model's confidence. This means a near-tie looks the same as a confident call. A probability distribution shows that confidence, which lets us flag uncertain cases or set thresholds when some mistakes are more serioud than others. However, it is harder to read and explain, and for k-NN the probabilities are rough, since they only show what share of the nearest neighbors belong to each class.

In [15]:
#Question 2 -----

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler

#loading data and EDA
lm = pd.read_csv("/content/land_mines.csv")

print("shape:", lm.shape)
print('missing values per column:', lm.isna().sum()) #no missing values

#summary stats for three features
print('feature summaries: ', lm[["voltage", "height", "soil"]].describe())

#looking at amount of each mine type
print('mine type counts:', lm['mine_type'].value_counts().sort_index())

#printing mean of each mine type and features
print('Mean:', lm.groupby('mine_type').mean())



#spliting the sample into training and test/validation sets
lm_X_raw = lm[['voltage', 'height', 'soil']]
lm_y = lm['mine_type']
(
    lm_X_train_raw,
    lm_X_valid_raw,
    lm_y_train,
    lm_y_valid,
) = train_test_split(
    lm_X_raw, lm_y, test_size=0.50, stratify=lm_y, random_state=65
)

lm_scaler = MinMaxScaler()
lm_X_train_scaled = lm_scaler.fit_transform(lm_X_train_raw) #computing training data min and max and scaling the validation features
lm_X_valid_scaled = lm_scaler.transform(lm_X_valid_raw) #using same values to scale the validation features

#building k-NN classifier and choosing k value
lm_ks = np.arange(1,51) #trying k values from 1-50
lm_valid_acc = [] #empty list to store each k's accucray on validation set
lm_train_acc = [] #empty list to store each k's accuracy on training set

for lm_k in lm_ks: #looping through for each k
  lm_candidate =KNeighborsClassifier(n_neighbors=int(lm_k)) #building k-NN model using k
  lm_candidate.fit(lm_X_train_scaled, lm_y_train) #learning from the training mines (k-NN storing them)
  lm_valid_hat = lm_candidate.predict(lm_X_valid_scaled) #guessing the mine type of every validation mine
  lm_train_hat = lm_candidate.predict(lm_X_train_scaled) #guess mine type for train set also
  lm_valid_acc.append(np.mean(lm_y_valid.to_numpy()== lm_valid_hat)) #saving correct validation guesses
  lm_train_acc.append(np.mean(lm_y_train.to_numpy()== lm_train_hat)) #saving correct training guesses

best_k = int(lm_ks[np.argmax(lm_valid_acc)]) #finding position of highest validation accuracy, finding that k value
print('Best k:', best_k)

#creating plot of training v validation accuracy for each k, used to choose k
plt.plot(lm_ks, lm_train_acc, label='Training accuracy')
plt.plot(lm_ks, lm_valid_acc, label='Validation accuracy')
plt.axvline(best_k, color='red', linestyle='--', label=f'best k = {best_k}')
plt.xlabel('k (number of neighbors)')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

#print a confusion table
lm_final = KNeighborsClassifier(n_neighbors=best_k) #refitting model using the best k
lm_final.fit(lm_X_train_scaled, lm_y_train) #train on the training data
lm_pred = lm_final.predict(lm_X_valid_scaled) #predict the validation data

#creating confusion table, rows = actual and columns = predicted
print(pd.crosstab(lm_y_valid, lm_pred, rownames=['actual'], colnames=['predicted']))

#printing overall accuracy
print('accuracy:', np.mean(lm_y_valid.to_numpy() == lm_pred))


shape: (338, 4)
missing values per column: voltage      0
height       0
soil         0
mine_type    0
dtype: int64
feature summaries:            voltage      height        soil
count  338.000000  338.000000  338.000000
mean     0.430634    0.508876    0.503550
std      0.195819    0.306043    0.344244
min      0.197734    0.000000    0.000000
25%      0.309737    0.272727    0.200000
50%      0.359516    0.545455    0.600000
75%      0.482628    0.727273    0.800000
max      0.999999    1.000000    1.000000
mine type counts: mine_type
1    71
2    70
3    66
4    66
5    65
Name: count, dtype: int64
Mean:             voltage    height      soil
mine_type                              
1          0.296463  0.495519  0.492958
2          0.721123  0.487013  0.508571
3          0.402408  0.527548  0.496970
4          0.345365  0.506887  0.503030
5          0.379598  0.530070  0.516923
Best k: 2
predicted   1   2   3   4  5
actual                      
1          27   0   5   2  2
2        

Question 2: written answers

3. Explain how you select k

I selected k by trying every value from 1 to 50, which covers small values that produce more flexible models and large values that produce smoother models. For each k, I fit and trained a k-NN classifer on the training data and measured its accuracy on the validation half, which the model had not seen. I chose the k with the highest validation accuracy because it performed best on the validation data (data it hadn't seen during training). This value was k = 2.

4. Print a confusion table for the optimal model, comparing predicted and actual class label on the test set. How accurate is it? Where is performance more or less accurate?

The model is overall 46.7% accurate. It performs best for the class 2 mine type predicting 31 out of 35 observations correctly. It also performed well for class 1, predicting 27 out of 36 observations correctly. It performs less accurately for classes 3, 4, and 5. It performed the worst for class 5, only correctly predicting 2 out of 32 observations.


5. Notice that you can have a lot of accurate predictions for a given type of mine, but still make a lot of mistakes. Please explain how you'd advise someone to actually use this predictive model in practice, given the errors that it tends to make.

Due to these errors, I would advise somoeone against treating the prediction label as a final answer. First, I would advise someone to use the model to narrow the possibilities while not making the final call for how to remove a mine. The model performs much better for some mine types than others, so predictions should be interpreted differently. For example, the model's predictions for type 2 are very reliable so it could be a trusted prediction but should still be backed up by additional metrics. However, for types 1, 3, 4, and 5 which have lower accuracy levels, the prediction could be interpretted as much less reliable and should be confirmed by additional evidence.

Second, I would confirm the models predictions with additional information and data before removing a mine. This could include another sensor reading, visual inspection, and judgment from experts. This would be especially critical for the 3, 4, and 5 types because the model makes the most mistake there.

Third, I would try to collect more data to train the model on. This would help the model become more accurate since there would be more training data for it to learn from.

## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:**
- **AI tool and model used:**

### *Prompts used

1. "Given this rubric and these instructions, help me improve my code and answers" I uploaded the rubric, instructions, my code, and the land mine data
2. [Paste follow-up prompt]

### *AI-proposed changes


[Paste the AI's concise numbered list verbatim.]

1. ...
2. ...

### *Evaluation of AI suggestions


| Change | Decision | Reason and verification |
|---|---|---|
| 1 | Accept / modify / reject | ... |
| 2 | Accept / modify / reject | ... |

### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

In [ ]:
# Your Phase 2 solution to the problem goes here.


### *Reflection on AI assistance
In your own words without generative AI.

[In 150–300 words, describe the main improvements, AI mistakes or limitations, how the final solution was verified, and any remaining concerns.]